# Dynamic System Identity v0.6
A four-qubit toy model with time-dependent couplings. Goal: detect cluster formation, merger and separation using dynamical diagnostics. This remains standard quantum dynamics; it is not evidence for a new physical time.

In [ ]:
import numpy as np, itertools, matplotlib.pyplot as plt
I=np.eye(2,dtype=complex); X=np.array([[0,1],[1,0]],complex)
def op(site):
    a=1
    for k in range(4): a=np.kron(a,X if k==site else I)
    return a
Xs=[op(i) for i in range(4)]
def Jmat(t):
    # two stable pairs; cross bridge turns on smoothly, stays high, then turns off
    on=1/(1+np.exp(-8*(t-2)))
    off=1/(1+np.exp(8*(t-6)))
    b=0.95*on*off
    J=np.zeros((4,4)); J[0,1]=J[1,0]=1.; J[2,3]=J[3,2]=.9
    for i in [0,1]:
        for j in [2,3]: J[i,j]=J[j,i]=b
    return J
def H(t):
    J=Jmat(t); h=np.zeros((16,16),complex)
    for i in range(4):
        for j in range(i+1,4): h += J[i,j]*(Xs[i]@Xs[j])
    return h
def step(psi,t,dt):
    w,v=np.linalg.eigh(H(t+dt/2)); U=(v*np.exp(-1j*w*dt))@v.conj().T
    return U@psi
def entropy(r):
    z=np.linalg.eigvalsh((r+r.conj().T)/2); z=z[z>1e-12]
    return float(-np.sum(z*np.log2(z)))
def reduced(psi,S):
    S=list(S); E=[i for i in range(4) if i not in S]
    a=psi.reshape([2]*4); a=np.transpose(a,S+E).reshape(2**len(S),2**len(E))
    return a@a.conj().T
def mutual_cut(psi,S):
    rS=reduced(psi,S); return 2*entropy(rS) # global state is pure
def boundary(J,S,K0=.1):
    S=set(S); E=set(range(4))-S
    win=sum(abs(J[i,j]) for i in S for j in S if i<j)
    wout=sum(abs(J[i,j]) for i in S for j in E)
    A=win/(win+wout+K0); return win,wout,A


## Evolve state and evaluate the {0,1}|{2,3} boundary

In [ ]:
dt=.01; ts=np.arange(0,8+dt,dt); psi=np.zeros(16,complex); psi[0]=1
A=[]; Ccut=[]; bridge=[]
for k,t in enumerate(ts):
    J=Jmat(t); A.append(boundary(J,(0,1))[2]); Ccut.append(mutual_cut(psi,(0,1))/4); bridge.append(J[0,2])
    if k<len(ts)-1: psi=step(psi,t,dt)
plt.figure(figsize=(9,4.5)); plt.plot(ts,A,label='A_partition {0,1}'); plt.plot(ts,Ccut,label='normalized cut MI'); plt.plot(ts,bridge,label='cross coupling'); plt.legend(); plt.xlabel('t'); plt.grid(alpha=.25); plt.show()


When cross coupling rises, the old {0,1}|{2,3} boundary loses autonomy while correlations across that cut can rise. When the bridge turns off, structural autonomy returns, but cross-cut quantum correlation need not instantly vanish. This creates a useful hysteresis-like distinction between interaction boundary and state memory without claiming new physics.

## Enumerate preferred pair partitions over time

In [ ]:
pairs=[(0,1),(0,2),(0,3)] # canonical 2|2 cuts
sample=np.linspace(0,8,81); psi=np.zeros(16,complex); psi[0]=1; cur=0.; out=[]
for target in sample:
    while cur+dt/2 < target:
        psi=step(psi,cur,dt); cur+=dt
    scores=[]
    for S in pairs:
        a=boundary(Jmat(target),S)[2]; c=mutual_cut(psi,S)/4
        # diagnostic only: structural autonomy rewarded, boundary entanglement penalized
        q=a*(1-c)
        scores.append((q,S,a,c))
    out.append((target,max(scores)))
for row in out[::10]: print(row)


## Identity-state labels (diagnostic, not ontology)
Use the structural boundary score only to label regimes: separated-cluster candidate when A({0,1}) is high; merged-network candidate when it is low because cross coupling is strong. Thresholds are illustrative and must not be interpreted as fundamental constants.

In [ ]:
def regime(a):
    if a>.65: return 'two-cluster candidate'
    if a<.35: return 'merged-network candidate'
    return 'transition'
for t in [0,1,2,3,4,5,6,7,8]: print(t,round(boundary(Jmat(t),(0,1))[2],3),regime(boundary(Jmat(t),(0,1))[2]))


## Scientific boundary
The appearance/disappearance of a preferred dynamical partition is generated entirely by an explicitly time-dependent standard Hamiltonian. It demonstrates a prototype of dynamic system identity, not temporal emergence. A new-time hypothesis would require an additional operational clock variable and a relational residual not reducible to this standard evolution.